# M6 · Aula 02 — Eixos, indexação, broadcasting e layout

Laboratório reproduzível em CPU para tornar explícitos os contratos de shape e armazenamento antes de estudar autograd.

**Dependências mínimas:** Python 3.10 e PyTorch 2.6.  
**Dados:** tensores sintéticos definidos no próprio notebook; nenhum download, segredo ou credencial.  
**Reprodutibilidade:** seed `20260902`, dtype `float64` e verificações automáticas.

Execute todas as células em ordem num kernel novo. A exceção de `view` é uma contraprova esperada e tratada.

## 1. Ambiente e contratos auxiliares

O laboratório não usa autograd: `requires_grad` permanece falso. O objetivo é saber quais números cada eixo representa e se duas expressões compartilham armazenamento.

In [ ]:
import platform
import warnings

import torch

SEED = 20260902
torch.manual_seed(SEED)
torch.set_default_dtype(torch.float64)

AUDIT = {}

def check(name, condition):
    assert bool(condition), f"Falha no contrato: {name}"
    AUDIT[name] = True

def check_shape(name, tensor, expected):
    check(name, tuple(tensor.shape) == tuple(expected))

print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "device": "cpu",
    "seed": SEED,
})

## 2. Eixos são uma convenção semântica

Usamos `B=2` exemplos, `T=3` posições e `D=4` atributos. PyTorch conhece `(2, 3, 4)`, mas os nomes `B`, `T` e `D` pertencem ao nosso contrato.

In [ ]:
B, T, D = 2, 3, 4
x = torch.arange(B * T * D, dtype=torch.float64).reshape(B, T, D)

check_shape("shape_BTD", x, (B, T, D))
check("numel_conservado", x.numel() == B * T * D)
check("indice_semantico", x[1, 2, 3].item() == 23.0)

print("x.shape =", tuple(x.shape))
print("x[1, 2, :] =", x[1, 2].tolist())

## 3. Indexação básica e avançada

Fatiamento básico normalmente cria uma *view*. Indexação por lista/tensor de inteiros cria uma cópia. A contraprova abaixo usa mutação controlada de clones para tornar a diferença observável.

In [ ]:
base_view = x.clone()
slice_view = base_view[:, 1:, :2]
slice_view[0, 0, 0] = -999.0

base_copy = x.clone()
advanced_copy = base_copy[:, torch.tensor([1, 2]), :2]
advanced_copy[0, 0, 0] = -777.0

check("slice_compartilha", base_view[0, 1, 0].item() == -999.0)
check("advanced_copia", base_copy[0, 1, 0].item() == x[0, 1, 0].item())
check_shape("slice_shape", slice_view, (2, 2, 2))
check_shape("advanced_shape", advanced_copy, (2, 2, 2))

print("mutação via slice chegou à base:", base_view[0, 1, 0].item())
print("base após mutar índice avançado:", base_copy[0, 1, 0].item())

## 4. Máscaras preservam apenas os elementos selecionados

Uma máscara booleana aplicada a um tensor pode achatar os eixos selecionados. Quando a estrutura por exemplo importa, retenha os índices ou use operações que declarem o eixo.

In [ ]:
scores = torch.tensor([[0.2, 0.9, -0.1], [0.7, 0.1, 0.8]])
mask = scores > 0.5
selected = scores[mask]
coordinates = mask.nonzero(as_tuple=False)

check_shape("mascara_shape", mask, (2, 3))
check_shape("selecao_achatada", selected, (3,))
check("valores_mascara", torch.equal(selected, torch.tensor([0.9, 0.7, 0.8])))
check("coordenadas_mascara", torch.equal(coordinates, torch.tensor([[0, 1], [1, 0], [1, 2]])))

print("selecionados =", selected.tolist())
print("coordenadas =", coordinates.tolist())

## 5. Reduções: qual eixo desaparece?

`dim=-1` reduz o último eixo. `keepdim=True` mantém esse eixo com tamanho 1, facilitando broadcasting explícito e auditoria de shapes.

In [ ]:
means_last = x.mean(dim=-1)
means_last_keep = x.mean(dim=-1, keepdim=True)
means_batch = x.mean(dim=0)

check_shape("media_ultimo_eixo", means_last, (B, T))
check_shape("media_keepdim", means_last_keep, (B, T, 1))
check_shape("media_lote", means_batch, (T, D))
check("keepdim_mesmos_valores", torch.equal(means_last_keep.squeeze(-1), means_last))

print("mean(dim=-1).shape =", tuple(means_last.shape))
print("mean(dim=0).shape =", tuple(means_batch.shape))

## 6. Broadcasting: alinhar dimensões pela direita

Para padronizar cada atributo ao longo de `T`, mantemos média e desvio com shape `(B, 1, D)`. O resultado coincide com a expansão manual, sem materializar cópias no broadcasting.

In [ ]:
mean_t = x.mean(dim=1, keepdim=True)
std_t = x.std(dim=1, keepdim=True, correction=0)
normalized = (x - mean_t) / std_t

manual = torch.empty_like(x)
for b in range(B):
    for t in range(T):
        for d in range(D):
            manual[b, t, d] = (x[b, t, d] - mean_t[b, 0, d]) / std_t[b, 0, d]

check_shape("estatisticas_B1D", mean_t, (B, 1, D))
check("broadcast_equivale_manual", torch.equal(normalized, manual))
check("media_normalizada_zero", torch.allclose(normalized.mean(dim=1), torch.zeros(B, D), atol=1e-15, rtol=0))
check("variancia_normalizada_um", torch.allclose(normalized.var(dim=1, correction=0), torch.ones(B, D), atol=1e-15, rtol=0))

print("erro máximo broadcast vs. laços =", (normalized - manual).abs().max().item())
print("médias por exemplo/atributo =", normalized.mean(dim=1).tolist())

## 7. Contraprova: broadcasting pode aceitar uma loss errada

Predições `(B,)` e alvos `(B,1)` formam todos os pares `(B,B)`. Shapes devem ser iguais antes de uma comparação elemento a elemento por exemplo.

In [ ]:
prediction = torch.tensor([1.0, 2.0, 3.0])
target_column = torch.tensor([[1.0], [2.0], [3.0]])
wrong_residual = prediction - target_column
wrong_mse = wrong_residual.square().mean()

target = target_column.squeeze(-1)
assert prediction.shape == target.shape
correct_mse = (prediction - target).square().mean()

check_shape("broadcast_indesejado_BB", wrong_residual, (3, 3))
check("mse_errada_4_sobre_3", wrong_mse.item() == 4 / 3)
check("mse_correta_zero", correct_mse.item() == 0.0)

print(f"MSE errada={wrong_mse.item():.6f}; MSE correta={correct_mse.item():.6f}")

## 8. `unsqueeze` e `squeeze`: declarar o eixo

`squeeze()` sem argumento remove **todos** os eixos de tamanho 1. Com lote unitário, isso pode apagar o eixo `B`. Prefira `squeeze(dim)` quando apenas um eixo deve desaparecer.

In [ ]:
single = torch.arange(4.0).reshape(1, 4, 1)
unsafe = single.squeeze()
safe = single.squeeze(-1)
restored = safe.unsqueeze(-1)

check_shape("squeeze_todos", unsafe, (4,))
check_shape("squeeze_controlado", safe, (1, 4))
check_shape("unsqueeze_restaura", restored, (1, 4, 1))
check("restauracao_exata", torch.equal(restored, single))

print("original, squeeze(), squeeze(-1):", tuple(single.shape), tuple(unsafe.shape), tuple(safe.shape))

## 9. `expand` versus `repeat`

`expand` cria uma view com stride zero no eixo expandido; `repeat` materializa repetições. Ambos podem mostrar os mesmos valores, mas têm armazenamento e custo distintos.

In [ ]:
bias = torch.tensor([[10.0, 20.0, 30.0]])
expanded = bias.expand(4, 3)
repeated = bias.repeat(4, 1)

check("expand_repeat_valores", torch.equal(expanded, repeated))
check("expand_stride_zero", expanded.stride() == (0, 1))
check("repeat_contiguo", repeated.is_contiguous())
check("expand_nao_contiguo", not expanded.is_contiguous())
check("expand_mesmo_storage", expanded.untyped_storage().data_ptr() == bias.untyped_storage().data_ptr())
check("repeat_storage_novo", repeated.untyped_storage().data_ptr() != bias.untyped_storage().data_ptr())

print("strides expand/repeat =", expanded.stride(), repeated.stride())
print("storage bytes base/expand/repeat =", bias.untyped_storage().nbytes(), expanded.untyped_storage().nbytes(), repeated.untyped_storage().nbytes())

## 10. Strides e transposição

Para um tensor contíguo `(2,3,4)`, avançar no último eixo move 1 elemento; no eixo intermediário, 4; no primeiro, 12. `permute` troca a interpretação dos eixos sem mover os dados.

In [ ]:
base = torch.arange(24).reshape(2, 3, 4)
permuted = base.permute(0, 2, 1)

check("stride_base", base.stride() == (12, 4, 1))
check_shape("permute_BDT", permuted, (2, 4, 3))
check("stride_permutado", permuted.stride() == (12, 1, 4))
check("permute_view_storage", permuted.untyped_storage().data_ptr() == base.untyped_storage().data_ptr())
check("permute_nao_contiguo", not permuted.is_contiguous())
check("semantica_indice_permuta", permuted[1, 3, 2].item() == base[1, 2, 3].item() == 23)

print("base:", tuple(base.shape), base.stride(), base.is_contiguous())
print("permuted:", tuple(permuted.shape), permuted.stride(), permuted.is_contiguous())

## 11. `view`, `reshape` e `contiguous`

`view` exige strides compatíveis. `reshape` pode devolver view ou cópia; código correto não depende dessa escolha. `contiguous()` preserva valores e materializa um layout contíguo quando necessário.

In [ ]:
view_rejected = False
try:
    permuted.view(2, 12)
except RuntimeError:
    view_rejected = True

reshaped = permuted.reshape(2, 12)
made_contiguous = permuted.contiguous()
view_after = made_contiguous.view(2, 12)

check("view_rejeitada_layout", view_rejected)
check_shape("reshape_shape", reshaped, (2, 12))
check("contiguous_valores", torch.equal(made_contiguous, permuted))
check("contiguous_layout", made_contiguous.is_contiguous())
check("view_apos_contiguous", torch.equal(view_after, reshaped))
check("contiguous_copiou", made_contiguous.untyped_storage().data_ptr() != permuted.untyped_storage().data_ptr())

print("view direta rejeitada =", view_rejected)
print("reshape/contiguous+view iguais =", torch.equal(reshaped, view_after))

## 12. Flatten sem misturar exemplos

Ao preparar imagens para uma MLP, preserve o lote: `(B,C,H,W) -> (B,C·H·W)`. `reshape(-1)` mistura exemplos num único vetor.

In [ ]:
images = torch.arange(2 * 1 * 2 * 3, dtype=torch.float64).reshape(2, 1, 2, 3)
flat = images.reshape(images.shape[0], -1)
wrong_flat = images.reshape(-1)

check_shape("flatten_preserva_B", flat, (2, 6))
check_shape("flatten_global", wrong_flat, (12,))
check("flatten_numel", flat.numel() == images.numel())
check("exemplo_zero_preservado", torch.equal(flat[0], torch.arange(6, dtype=torch.float64)))

print("images -> flat -> global:", tuple(images.shape), tuple(flat.shape), tuple(wrong_flat.shape))

## 13. Integração: transformação afim em dados com dois eixos estruturais

Uma camada afim atua no último eixo de `(B,T,D)`. Achatamos somente `B·T`, aplicamos a mesma equação e restauramos os eixos; o resultado deve coincidir com `x @ W + b`.

In [ ]:
generator = torch.Generator().manual_seed(SEED)
x_fixture = torch.randn(5, 7, 4, generator=generator)
W = torch.randn(4, 3, generator=generator)
b = torch.randn(3, generator=generator)

direct = x_fixture @ W + b
flat_result = (x_fixture.reshape(-1, 4) @ W + b).reshape(5, 7, 3)
max_error = (direct - flat_result).abs().max().item()

check_shape("afim_BTH", direct, (5, 7, 3))
check("afim_equivalente", max_error == 0.0)
check("afim_finita", torch.isfinite(direct).all())

print(f"erro máximo entre formas da afim = {max_error:.3e}")

## 14. Auditoria final

Os contratos abaixo abrangem valores, shapes, aliasing, strides, contiguidade e contraprovas. O notebook deve terminar sem warnings inesperados.

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    probe = torch.ones(2, 1) + torch.arange(3).reshape(1, 3)

check_shape("probe_broadcast", probe, (2, 3))
check("sem_warnings_inesperados", len(caught) == 0)

expected_contracts = 53
check("quantidade_pre_final", len(AUDIT) == expected_contracts)

print(f"{len(AUDIT)}/{len(AUDIT)} contratos aprovados")
print("Nenhum warning inesperado.")

## Leitura dos resultados

- A indexação básica compartilhou armazenamento; a avançada criou cópia.
- Broadcasting e laços explícitos produziram exatamente os mesmos valores na fixture.
- A MSE errada foi `1.333333`, apesar de predições e alvos conterem os mesmos números.
- `expand(4,3)` usou o armazenamento de 24 bytes da base; `repeat(4,1)` usou 96 bytes no ambiente de referência.
- A permutação mudou strides de `(12,4,1)` para `(12,1,4)` e exigiu cópia antes de `view`.
- A transformação afim direta e a forma achatada/restaurada coincidiram exatamente.

Esses testes verificam o contrato da fixture em CPU; não são benchmark de desempenho e não garantem que `reshape` sempre copie ou sempre compartilhe em outros layouts.